# Proteomics Protein Analysis Visualizations

This notebook creates visualizations from the merged proteomics data. It can be run independently after the main `multi_dataset_protein_analysis.ipynb` notebook has been executed.

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from matplotlib_venn import venn2, venn3

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Load Data from CSV Files

In [ ]:
# Load the saved data from CSV files
merged_df = pd.read_csv('protein_analysis_results/merged_raw_data.csv')
protein_stats = pd.read_csv('protein_analysis_results/protein_statistics.csv', index_col='Protein ID')
summary_dataset = pd.read_csv('protein_analysis_results/summary_by_dataset.csv')
summary_condition = pd.read_csv('protein_analysis_results/summary_by_condition.csv')

print("Data loaded successfully")
print(f"Total proteins: {len(protein_stats)}")
print(f"Total rows in merged data: {len(merged_df)}")
print(f"Columns in merged_df: {list(merged_df.columns)[:10]}...")  # Show first 10 columns

## Summary Statistics Visualizations

In [ ]:
## Proteins and Peptides by Dataset
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Proteins by dataset
ax1.bar(summary_dataset['Dataset'], summary_dataset['Unique Proteins'], color='skyblue')
ax1.set_title('Unique Proteins by Dataset')
ax1.set_ylabel('Number of Proteins')
ax1.tick_params(axis='x', rotation=45)

# Peptides by dataset
ax2.bar(summary_dataset['Dataset'], summary_dataset['Unique Peptides'], color='lightgreen')
ax2.set_title('Unique Peptides by Dataset')
ax2.set_ylabel('Number of Peptides')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Proteins and Peptides by Condition
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Proteins by condition
ax1.barh(summary_condition['Condition'], summary_condition['Unique Proteins'], color='coral')
ax1.set_title('Unique Proteins by Condition')
ax1.set_xlabel('Number of Proteins')

# Peptides by condition
ax2.barh(summary_condition['Condition'], summary_condition['Unique Peptides'], color='gold')
ax2.set_title('Unique Peptides by Condition')
ax2.set_xlabel('Number of Peptides')

plt.tight_layout()
plt.show()

## Individual Protein Statistics

In [ ]:
# Distribution of Protein Metrics
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Number of datasets per protein
ax1.hist(protein_stats['Num Datasets'], bins=range(1, int(protein_stats['Num Datasets'].max())+2), alpha=0.7, edgecolor='black', color='steelblue')
ax1.set_title('Distribution of Number of Datasets per Protein')
ax1.set_xlabel('Number of Datasets')
ax1.set_ylabel('Frequency')

# Number of conditions per protein
ax2.hist(protein_stats['Num Conditions'], bins=range(1, int(protein_stats['Num Conditions'].max())+2), alpha=0.7, edgecolor='black', color='coral')
ax2.set_title('Distribution of Number of Conditions per Protein')
ax2.set_xlabel('Number of Conditions')
ax2.set_ylabel('Frequency')

# Total identifications
ax3.hist(protein_stats['Total Identifications'], bins=50, alpha=0.7, edgecolor='black', log=True, color='lightgreen')
ax3.set_title('Distribution of Total Identifications per Protein (log scale)')
ax3.set_xlabel('Total Identifications')
ax3.set_ylabel('Frequency (log)')

# Scatter plot: Datasets vs Conditions, sized by Identifications
scatter = ax4.scatter(protein_stats['Num Datasets'], protein_stats['Num Conditions'], 
                     s=np.log10(protein_stats['Total Identifications']+1)*50, 
                     alpha=0.6, c=protein_stats['Total Identifications'], 
                     cmap='viridis', edgecolors='black', linewidth=0.5)
ax4.set_title('Datasets vs Conditions per Protein\n(size and color = log identifications)')
ax4.set_xlabel('Number of Datasets')
ax4.set_ylabel('Number of Conditions')
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Total Identifications')

plt.tight_layout()
plt.show()

## Overlap Analysis Setup

In [ ]:
# Dataset Protein Sets
dataset_proteins = {}
for dataset in merged_df['Dataset'].unique():
    dataset_proteins[dataset] = set(merged_df[merged_df['Dataset'] == dataset]['Protein ID'].unique())

print("All unique proteins per dataset (from merged_df):")
for ds, proteins in sorted(dataset_proteins.items()):
    print(f"{ds}: {len(proteins)} proteins")

# Condition Protein Sets
condition_proteins = {}
for condition in merged_df['Condition'].unique():
    condition_proteins[condition] = set(merged_df[merged_df['Condition'] == condition]['Protein ID'].unique())

print("\nAll unique proteins per condition (from merged_df):")
for cond, proteins in sorted(condition_proteins.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{cond}: {len(proteins)} proteins")

## Dataset Overlaps - Venn Diagrams

In [ ]:
# Since matplotlib-venn doesn't support 4 sets, we'll show pairwise comparisons
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

datasets = list(dataset_proteins.keys())
for i in range(len(datasets)):
    for j in range(i+1, len(datasets)):
        ax_idx = (i * (len(datasets)-1)) + (j - i - 1)
        if ax_idx < 6:  # Only 6 subplots
            venn2([dataset_proteins[datasets[i]], dataset_proteins[datasets[j]]],
                  set_labels=[datasets[i], datasets[j]], ax=axes[ax_idx])
            axes[ax_idx].set_title(f'{datasets[i]} vs {datasets[j]}')

# Hide unused subplots
for k in range(ax_idx + 1, 6):
    axes[k].set_visible(False)

plt.tight_layout()
plt.show()

## Condition Overlaps - Venn Diagrams

In [ ]:
# Show pairwise comparisons for the top conditions by protein count
top_conditions = sorted(condition_proteins.items(), key=lambda x: len(x[1]), reverse=True)[:6]
top_condition_names = [c[0] for c in top_conditions]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (i, j) in enumerate([(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]):
    if idx < len(axes) and i < len(top_condition_names) and j < len(top_condition_names):
        cond_i = top_condition_names[i]
        cond_j = top_condition_names[j]
        venn2([condition_proteins[cond_i], condition_proteins[cond_j]],
              set_labels=[cond_i, cond_j], ax=axes[idx])
        axes[idx].set_title(f'{cond_i} vs {cond_j}')

# Hide unused subplots
for k in range(idx + 1, 6):
    axes[k].set_visible(False)

plt.tight_layout()
plt.suptitle('Protein Overlaps Between Top Conditions', y=1.00)
plt.show()

## Interactive Visualizations

In [ ]:
# Scatter Plot: Proteins by Number of Peptides and Datasets
fig = px.scatter(protein_stats.reset_index(),
                 x='Num Datasets',
                 y='Total Identifications',
                 color='Num Conditions',
                 size='Total Identifications',
                 hover_name='Protein ID',
                 title='Protein Identifications: Datasets vs Total Identifications',
                 labels={'Num Datasets': 'Number of Datasets',
                        'Total Identifications': 'Total Identifications',
                        'Num Conditions': 'Number of Conditions'},
                 log_y=True)

fig.update_layout(width=1000, height=600)
fig.show()

## Top Proteins by Identifications
top_proteins = protein_stats.nlargest(20, 'Total Identifications').reset_index()

# Extract protein names for better display
def extract_protein_name(protein_id):
    parts = str(protein_id).split()
    if len(parts) > 1:
        return ' '.join(parts[1:3])  # Take first two words after accession
    return str(protein_id)

top_proteins['Protein Name'] = top_proteins['Protein ID'].apply(extract_protein_name)

fig = px.bar(top_proteins,
             x='Total Identifications',
             y='Protein Name',
             orientation='h',
             title='Top 20 Proteins by Total Identifications',
             labels={'Total Identifications': 'Total Identifications',
                    'Protein Name': 'Protein'})

fig.update_layout(width=1000, height=800)
fig.show()